# DATATHON 2026 — THE GRIDBREAKER
**Hosted by:** VinTelligence - VinUniversity Data Science & AI Club

---

## THÔNG TIN NHÓM
- **Tên đội thi**: Last Dance
- **Mã đội thi**: kVmzJpHWUFT6mv82wG4U
- **Thành viên**: 

| STT | Họ và tên        |  Vai trò   | Email |
| --- | ---------------- | ---------- | ----- |
| 1   | Bùi Lê Khôi      | Đội trưởng |       |
| 2   | Nguyễn Hà Anh    | Thành viên |       |
| 3   | Bùi Công Mậu     | Thành viên |       |
| 4   | Thái Hữu Thọ     | Thành viên |       |

---

## GIỚI THIỆU FILE NOTEBOOK

- **Mục đích**: Tiền xử lí sâu trước khi train mô hình - **Phần 3 — Mô hình Dự báo Doanh thu (Sales Forecasting)**

- **Dữ liệu sử dụng**:
    - Dữ liệu đầu vào là file đầu ra sau khi chạy file LD_02_DA_01_Prepare.
    
---

Bài làm được trình bày từ sau dòng này.

---
---

### Import các libs và dependencies

In [23]:
import re
import os
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl as pxl
import seaborn as sns
import statsmodels.api as sm
import datetime as dt

from datetime import timedelta
from scipy import stats
from math import sqrt

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, pairwise_distances, silhouette_score, precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

import warnings
warnings.filterwarnings("ignore")

# Ensure plots appear in the notebook
%matplotlib inline

# Print last updated timestamp
import time
print(f"Cập nhật lần cuối vào thời điểm {time.asctime()}")

Cập nhật lần cuối vào thời điểm Thu Apr 30 04:48:57 2026


---

### Đọc dữ liệu

In [24]:
# Đọc tất cả data từ data/processed, lưu vào các dataframe trong một dictionary

data_dir = '../data/processed'
data_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
data_frames = {}
for file in data_files:
    file_path = os.path.join(data_dir, file)
    df_name = os.path.splitext(file)[0]
    data_frames[df_name] = pd.read_csv(file_path)

# Kiểm tra dữ liệu đã được đọc thành công
for name, df in data_frames.items():
    print(f"DataFrame '{name}' has {df.shape[0]} rows and {df.shape[1]} columns.")

DataFrame 'customers' has 121930 rows and 7 columns.
DataFrame 'geography' has 39948 rows and 4 columns.
DataFrame 'inventory' has 60247 rows and 17 columns.
DataFrame 'orders' has 646945 rows and 8 columns.
DataFrame 'order_items' has 714669 rows and 7 columns.
DataFrame 'payments' has 646945 rows and 4 columns.
DataFrame 'products' has 2412 rows and 8 columns.
DataFrame 'promotions' has 50 rows and 10 columns.
DataFrame 'returns' has 39939 rows and 7 columns.
DataFrame 'reviews' has 113551 rows and 7 columns.
DataFrame 'sales' has 3833 rows and 3 columns.
DataFrame 'sample_submission' has 548 rows and 3 columns.
DataFrame 'shipments' has 566067 rows and 4 columns.
DataFrame 'web_traffic' has 3652 rows and 7 columns.


In [25]:
# # Kiểm tra thông tin chi tiết về từng DataFrame
# for name, df in data_frames.items():
#     print(f"\nDataFrame '{name}' info:")
#     print(df.info())

In [26]:
# Check số giá trị unique của các cột object trong từng DataFrame
for name, df in data_frames.items():
    print(f"\nDataFrame '{name}' unique values for object columns:")
    for col in df.select_dtypes(include=['object']).columns:
        unique_values = df[col].unique()
        print(f"  Column '{col}': {len(unique_values)} unique values")


DataFrame 'customers' unique values for object columns:
  Column 'city': 42 unique values
  Column 'signup_date': 3941 unique values
  Column 'gender': 3 unique values
  Column 'age_group': 5 unique values
  Column 'acquisition_channel': 6 unique values

DataFrame 'geography' unique values for object columns:
  Column 'city': 42 unique values
  Column 'region': 3 unique values
  Column 'district': 39 unique values

DataFrame 'inventory' unique values for object columns:
  Column 'snapshot_date': 126 unique values
  Column 'product_name': 1465 unique values
  Column 'category': 4 unique values
  Column 'segment': 8 unique values

DataFrame 'orders' unique values for object columns:
  Column 'order_date': 3833 unique values
  Column 'order_status': 6 unique values
  Column 'payment_method': 5 unique values
  Column 'device_type': 3 unique values
  Column 'order_source': 6 unique values

DataFrame 'order_items' unique values for object columns:
  Column 'promo_id': 51 unique values
  Col

In [27]:
# Chuyển các cột dưới đây thành datetime
# Bảng customers: signup_date
# Bảng inventory: snapshot_date
# Bảng orders: order_date
# Bảng promotions: start_date, end_date
# Bảng returns: return_date
# Bảng reviews: review_date
# Bảng sales: Date
# Bảng shipments: ship_date, delivery_date
# Bảng web_traffic: date

date_columns = {
    'customers': ['signup_date'],
    'inventory': ['snapshot_date'],
    'orders': ['order_date'],
    'promotions': ['start_date', 'end_date'],
    'returns': ['return_date'],
    'reviews': ['review_date'],
    'sales': ['Date'],
    'sample_submission': ['Date'],
    'shipments': ['ship_date', 'delivery_date'],
    'web_traffic': ['date']
}

for df_name, cols in date_columns.items():
    for col in cols:
        data_frames[df_name][col] = pd.to_datetime(data_frames[df_name][col], errors='coerce')

### Thực hiện Encoding dữ liệu

In [28]:
# Check các giá trị object của mỗi cột object trong mỗi bảng để xem có những giá trị nào.
for name, df in data_frames.items():
    print(f"\nDataFrame '{name}' object columns:")
    for col in df.select_dtypes(include=['object']).columns:
        unique_values = df[col].unique()
        print(f"  Column '{col}' has {len(unique_values)} unique values: {unique_values[:10]}...")


DataFrame 'customers' object columns:
  Column 'city' has 42 unique values: ['Hai Phong' 'Phu Ly' 'Viet Tri' 'Bac Giang' 'Lao Cai' 'Ha Long' 'Cam Pha'
 'Son Tay' 'Ninh Binh' 'Bac Ninh']...
  Column 'gender' has 3 unique values: ['Female' 'Male' 'Non-binary']...
  Column 'age_group' has 5 unique values: ['35-44' '45-54' '18-24' '55+' '25-34']...
  Column 'acquisition_channel' has 6 unique values: ['Social media' 'Email campaign' 'Organic search' 'Referral' 'Direct'
 'Paid search']...

DataFrame 'geography' object columns:
  Column 'city' has 42 unique values: ['Hai Phong' 'Phu Ly' 'Viet Tri' 'Bac Giang' 'Lao Cai' 'Ha Long' 'Cam Pha'
 'Son Tay' 'Ninh Binh' 'Bac Ninh']...
  Column 'region' has 3 unique values: ['East' 'Central' 'West']...
  Column 'district' has 39 unique values: ['District 13' 'District 04' 'District 06' 'District 05' 'District 03'
 'District 01' 'District 14' 'District 15' 'District 02' 'District 10']...

DataFrame 'inventory' object columns:
  Column 'product_name' ha

In [29]:
# Encoding các cột sau đây:
# Bảng customers: gender, age_group
# Bảng inventory: category, segment
# Bảng orders: order_status, payment_method, device_type, order_source
# Bảng payments: payment_method
# Bảng products: category, segment, size, color
# Bảng promotions: promo_type, promo_channel
# Bảng returns: return_reason
# Bảng web_traffic: traffic_source

In [30]:
# Ordinal mapping
age_group_map = {
    '18-24': 1,
    '25-35': 2,
    '35-44': 3,
    '45-54': 4,
    '55+': 5
}

size_map = {
    'S': 1,
    'M': 2,
    'L': 3,
    'XL': 4,
}

data_frames['customers']['age_group'] = data_frames['customers']['age_group'].map(age_group_map)
data_frames['products']['size'] = data_frames['products']['size'].map(size_map)

In [31]:
# Nominal mapping bằng LabelEncoder
le = LabelEncoder()

# Đồng bộ các cột xuất hiện ở nhiều bảng

# payment_method (orders & payments)
if 'orders' in data_frames and 'payments' in data_frames:
    all_payments = pd.concat([data_frames['orders']['payment_method'], 
                             data_frames['payments']['payment_method']]).astype(str).unique()
    le.fit(all_payments)
    data_frames['orders']['payment_method'] = le.transform(data_frames['orders']['payment_method'].astype(str))
    data_frames['payments']['payment_method'] = le.transform(data_frames['payments']['payment_method'].astype(str))

# category (products & inventory)
if 'products' in data_frames and 'inventory' in data_frames:
    all_cats = pd.concat([data_frames['products']['category'], 
                          data_frames['inventory']['category']]).astype(str).unique()
    le.fit(all_cats)
    data_frames['products']['category'] = le.transform(data_frames['products']['category'].astype(str))
    data_frames['inventory']['category'] = le.transform(data_frames['inventory']['category'].astype(str))

# segment (products & inventory)
if 'products' in data_frames and 'inventory' in data_frames:
    all_segs = pd.concat([data_frames['products']['segment'], 
                          data_frames['inventory']['segment']]).astype(str).unique()
    le.fit(all_segs)
    data_frames['products']['segment'] = le.transform(data_frames['products']['segment'].astype(str))
    data_frames['inventory']['segment'] = le.transform(data_frames['inventory']['segment'].astype(str))


# Encode các cột độc lập còn lại
simple_encode_tasks = {
    'customers': ['gender'],
    'orders': ['order_status', 'device_type', 'order_source'],
    'products': ['color'],
    'promotions': ['promo_type', 'promo_channel'],
    'returns': ['return_reason'],
    'web_traffic': ['traffic_source']
}

for df_name, cols in simple_encode_tasks.items():
    if df_name in data_frames:
        for col in cols:
            data_frames[df_name][col] = le.fit_transform(data_frames[df_name][col].astype(str))

print("Encoding hoàn tất trên toàn bộ dictionary data_frames!")

Encoding hoàn tất trên toàn bộ dictionary data_frames!


In [32]:
#Check kiểu dữ liệu của các cột trong từng DataFrame
for name, df in data_frames.items():
    print(f"\nDataFrame '{name}' column data types:")
    print(df.dtypes)


DataFrame 'customers' column data types:
customer_id                     int64
zip                             int64
city                           object
signup_date            datetime64[ns]
gender                          int64
age_group                     float64
acquisition_channel            object
dtype: object

DataFrame 'geography' column data types:
zip          int64
city        object
region      object
district    object
dtype: object

DataFrame 'inventory' column data types:
snapshot_date        datetime64[ns]
product_id                    int64
stock_on_hand                 int64
units_received                int64
units_sold                    int64
stockout_days                 int64
days_of_supply              float64
fill_rate                   float64
stockout_flag                 int64
overstock_flag                int64
reorder_flag                  int64
sell_through_rate           float64
product_name                 object
category                      int64


In [33]:
# --- BỔ SUNG ĐỂ TỐI ƯU HÓA CHO NHIỀU LOẠI MODEL ---

# 1. Trích xuất đặc trưng từ cột Date trong bảng sales (Quan trọng cho Forecasting)
if 'sales' in data_frames:
    df_sales = data_frames['sales']
    df_sales['month'] = df_sales['Date'].dt.month
    df_sales['day_of_week'] = df_sales['Date'].dt.dayofweek
    df_sales['is_weekend'] = df_sales['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    df_sales['day'] = df_sales['Date'].dt.day
    print("Đã trích xuất đặc trưng thời gian cho bảng sales.")

# 2. Sử dụng One-Hot Encoding cho các biến định danh có ít nhóm (Low Cardinality)
# Điều này giúp các model như Linear Regression hoặc KNN không bị nhầm lẫn về thứ tự.
ohe_cols = {
    'customers': ['gender'],
    'orders': ['device_type', 'order_source'],
    'web_traffic': ['traffic_source']
}

for df_name, cols in ohe_cols.items():
    if df_name in data_frames:
        # Sử dụng get_dummies để tạo các cột binary
        data_frames[df_name] = pd.get_dummies(data_frames[df_name], columns=cols, prefix=cols)
        print(f"Đã áp dụng One-Hot Encoding cho {df_name}: {cols}")

# 3. Xử lý giá trị âm hoặc bất thường (nếu có) trước khi train
# Đối với doanh thu hoặc giá, đôi khi log-transform sẽ giúp model dễ học hơn
# data_frames['sales']['Sales_Log'] = np.log1p(data_frames['sales']['Sales'])

print("Hoàn tất bổ sung thuộc tính. Dữ liệu đã sẵn sàng để thử nghiệm với đa dạng Model!")

# Lưu vào folder data/training
training_dir = '../data/training'
if not os.path.exists(training_dir):
    os.makedirs(training_dir)
for name, df in data_frames.items():
    df.to_csv(os.path.join(training_dir, f"{name}.csv"), index=False)

Đã trích xuất đặc trưng thời gian cho bảng sales.
Đã áp dụng One-Hot Encoding cho customers: ['gender']
Đã áp dụng One-Hot Encoding cho orders: ['device_type', 'order_source']
Đã áp dụng One-Hot Encoding cho web_traffic: ['traffic_source']
Hoàn tất bổ sung thuộc tính. Dữ liệu đã sẵn sàng để thử nghiệm với đa dạng Model!
